# Purpose:
- Generate czstack id - hcr id matching table from the last landmarks file

In [15]:
from pathlib import Path
import pandas as pd
import numpy as np
import os
import json

from manual_coreg_utils import get_ids_from_landmarks

DATA_DIR = Path('/root/capsule/data/')
%load_ext autoreload
%autoreload 2

In [2]:
def get_czstack_id(x):
    if x.startswith('cz'):
        return int(x.split('-')[0][2:])
    else:
        return -1


def get_hcr_id(x):
    if x.startswith('cz'):
        return int(x.split('-')[1][3:])
    else:
        return -1

In [33]:
subject_id = 788406
czstack_date = '2025-07-18'
save_dir = Path(f'/root/capsule/scratch/{subject_id}_{czstack_date}_coreg_cpsam/')

final_iter_num = 4
final_landmarks_file = save_dir / f'{subject_id}_{czstack_date}_landmarks_matched_ext_iter{final_iter_num}_qced.csv'
next_to_final_landmarks_file = save_dir / f'{subject_id}_{czstack_date}_landmarks_matched_ext_iter{final_iter_num+1}.csv'
assert final_landmarks_file.exists(), f'{final_landmarks_file} does not exist'
assert not next_to_final_landmarks_file.exists(), f'{next_to_final_landmarks_file} does exist'

In [26]:
save_fn = save_dir / f'{subject_id}_coreg_table.csv'

final_qced_landmarks = pd.read_csv(final_landmarks_file)
columns = ['ids', 'active', 'czstack_x', 'czstack_y', 'czstack_z', 'hcr_x', 'hcr_y', 'hcr_z']
final_qced_landmarks.columns = columns
prev_qced_ids = final_qced_landmarks[final_qced_landmarks.ids.str.startswith('qced')].ids.values
prev_qced_ids = [qcid.replace('qced_','') for qcid in prev_qced_ids]
final_qced_landmarks.loc[final_qced_landmarks.ids.str.startswith('qced'), 'active'] = True
final_qced_landmarks.loc[final_qced_landmarks.ids.str.startswith('qced'), 'ids'] = prev_qced_ids
num_roi_points = sum(final_qced_landmarks.ids.str.startswith('cz'))

final_active_landmarks = final_qced_landmarks.query('active').copy()

matched_landmarks = final_active_landmarks.copy()
matched_landmarks['czstack_id'] = matched_landmarks.ids.apply(get_czstack_id)
matched_landmarks['hcr_id'] = matched_landmarks.ids.apply(get_hcr_id)
matched_landmarks = matched_landmarks[matched_landmarks.czstack_id != -1]
assert matched_landmarks.active.sum() == len(matched_landmarks)
assert len(set(matched_landmarks.czstack_id)) == len(matched_landmarks)
assert len(set(matched_landmarks.hcr_id)) == len(matched_landmarks)
matched_landmarks = matched_landmarks.sort_values(by='czstack_id')[['czstack_id', 'hcr_id']].reset_index(drop=True)

# final assurance with alternative method to get ids
czstack_ids, hcr_ids = get_ids_from_landmarks(final_active_landmarks)
check_df = pd.DataFrame({'czstack_id': czstack_ids, 'hcr_id': hcr_ids})
check_df = check_df.sort_values(by='czstack_id').reset_index(drop=True)
assert check_df.equals(matched_landmarks)

# saving
matched_landmarks.to_csv(save_fn, index=False)

# report
print(f'Total number of czstack ids: {num_roi_points}')
print(f'Number of matched landmarks: {len(matched_landmarks)}')
print(f'Matching rate: {len(matched_landmarks)/num_roi_points:.2%}')

Total number of czstack ids: 931
Number of matched landmarks: 787
Matching rate: 84.53%


# Create New Data